# §27 — Cubic'i GELİŞTİR: eta (plato ölçeği) taraması

**Hipotez (mekanistik, koşudan önce yazıldı).** Cubic'in retention'ı
λ = 1/√(1+2η·z²); plato ölçeği **z\* ≈ 1/√(2η)**. Kod bugüne kadar η'yı
`logspace(-4,-2)` ile **sabit** başlattı → z\* ≈ 7–70. Ama bu görevin taşıma
menzili 256–4096+ token. Yani cubic'in platosu göreve göre **~2 mertebe kısa**
ayarlanmış olabilir — bu, cubic'in §15h (yoğun graft) ve §26b'de belirgin
üstünlük gösterememesinin somut adayı. Küçük η → uzun plato.

| log10(η) | z\* ≈ 1/√(2η) |
|---|---|
| −2 | 7 |
| −4 (varsayılan üst) | 71 |
| −6 | 707 |
| −8 | 7 071 |

**§26b'nin dersi uygulanıyor: birincil metrik DÜŞÜK VARYANSLI olmalı.** Probe
doğruluğu seed gürültüsünde boğuldu (exp %8→%70 savruldu). Bu yüzden birincil
endpoint = **eğitim-sonu cross-chunk doğrulama kaybı** (yüzlerce batch üzerinden
öğrenilen; §26b'de 3/3 seed'de tutarlıydı). Probe accuracy ikincil.

**ÖN-KAYITLI KRİTERLER.** Kollar: η ∈ {(−4,−2) *mevcut/kontrol*, (−6,−4), (−8,−6)},
3 seed; exp aynı araçta referans. 
- **CUBIC GELİŞTİ:** bir η kolu, varsayılana göre cross-chunk kaybı **≥0.3 nat**
  düşürüyorsa VE 3/3 seed'de varsayılandan iyiyse → η yanlış ayarlıymış; yeni
  varsayılan olarak önerilir, ardından graft rejiminde (§15h tekrarı) sınanır.
- **KISMİ:** ≥0.15 nat ve 2/3 seed → yön doğru, daha geniş tarama gerekir.
- **NULL:** hiçbir kol ≥0.15 nat kazandırmıyorsa → η ayarı sorun değildi; cubic'in
  sınırı başka yerde. Dürüstçe böyle yazılır.

Tamamen **CPU'da** koşar (GPU kotası yakmaz). ~9 eğitim kolu.


In [ ]:
# --- 1. KURULUM ---
import os, subprocess, sys, re, json, statistics as st
BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else ('/content' if os.path.exists('/content') else '.')
REPO = os.path.join(BASE,'HFP')
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/kayra-hn/HFP.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull'],check=True)
ROOT = os.path.join(BASE,'eta_sweep'); os.makedirs(ROOT, exist_ok=True)
print('repo:', REPO, '| cikti:', ROOT)
import math
for le in (-2,-4,-6,-8): print(f'  log10(eta)={le:>3} -> z* ~ {1/math.sqrt(2*10**le):.0f}')

In [ ]:
# --- 2. TARAMA (her kol AYRI dizin -> checkpoint cakismasi yok) ---
ARMS = [('cubic_flux_chunked', -4.0, -2.0, 'cubic_eta_default'),   # kontrol (mevcut)
        ('cubic_flux_chunked', -6.0, -4.0, 'cubic_eta_m6_m4'),
        ('cubic_flux_chunked', -8.0, -6.0, 'cubic_eta_m8_m6'),
        # [§27b] Veri gradyanini takip: kucuk eta KOTULESTIRDI (uzun plato -> girisim
        # birikiyor). O halde ters yon: BUYUK eta = kisa plato = hizli girisim temizligi.
        ('cubic_flux_chunked', -2.0,  0.0, 'cubic_eta_m2_0'),      # z* ~ 0.7..7
        ('cubic_flux_chunked',  0.0,  2.0, 'cubic_eta_0_p2'),      # z* ~ 0.07..0.7 (agresif)
        ('exp',                 None, None, 'exp_reference')]
SEEDS=[0,1,2]
BASE_ENV = {**os.environ, 'PYTHONPATH': REPO,
            'CC_CARRY_MAX':'16','CC_STEPS':'1200','CC_CTX':'256','CC_P':'6',
            'CC_DIST_EVERY':'64','CC_BS':'8','CC_GAPS':'256','CC_TRIALS':'60'}
RE_VER = re.compile(r'cross-chunk dogrulama loss:\s*([0-9.]+)')
RE_ACC = re.compile(r"FINAL acc%/logprob.*?\{256:\s*\(([0-9.]+)")
results={}
for mode, lo, hi, tag in ARMS:
    ck = os.path.join(ROOT, tag); os.makedirs(ck, exist_ok=True)
    env = {**BASE_ENV, 'HFP_CKPT_DIR': ck}
    if lo is not None:
        env['HFP_ETA_LOG_MIN']=str(lo); env['HFP_ETA_LOG_MAX']=str(hi)
    else:
        env.pop('HFP_ETA_LOG_MIN',None); env.pop('HFP_ETA_LOG_MAX',None)
    results[tag]={'loss':[], 'acc':[]}
    for s in SEEDS:
        cache=os.path.join(ck,f'result_s{s}.json')
        if os.path.exists(cache):
            d=json.load(open(cache)); results[tag]['loss'].append(d['loss']); results[tag]['acc'].append(d['acc'])
            print(f'[atla] {tag} s{s}: loss {d["loss"]:.3f} acc {d["acc"]:.1f}%'); continue
        print(f'\n=== {tag} s{s} (eta {lo}..{hi}) ===', flush=True)
        r = subprocess.run([sys.executable,'review_scripts/carry_curriculum.py',mode,str(s),'6000'],
                           cwd=REPO, env=env, capture_output=True, text=True)
        out=r.stdout+r.stderr
        mv=RE_VER.search(out); ma=RE_ACC.search(out)
        if not mv:
            print('  UYARI: kayip okunamadi. Son satirlar:'); print('\n'.join(out.strip().splitlines()[-6:])); continue
        loss=float(mv.group(1)); acc=float(ma.group(1)) if ma else float('nan')
        json.dump({'loss':loss,'acc':acc}, open(cache,'w'))
        results[tag]['loss'].append(loss); results[tag]['acc'].append(acc)
        print(f'  -> cross-chunk dogrulama loss {loss:.3f} | streaming probe @256 {acc:.1f}%', flush=True)
json.dump(results, open(os.path.join(ROOT,'eta_sweep_results.json'),'w'), indent=2)
print('\nTARAMA TAMAM ->', os.path.join(ROOT,'eta_sweep_results.json'))

In [ ]:
# --- 3. ON-KAYITLI HUKUM (§27) ---
import statistics as st
def m(v): return st.mean(v) if v else float('nan')
base = results.get('cubic_eta_default',{}).get('loss',[])
print(f"{'kol':>20} {'cross-chunk loss (birincil)':>32} {'probe@256 (ikincil)':>24}")
print('-'*80)
for tag in ['cubic_eta_0_p2','cubic_eta_m2_0','cubic_eta_default','cubic_eta_m6_m4','cubic_eta_m8_m6','exp_reference']:
    L=results.get(tag,{}).get('loss',[]); A=results.get(tag,{}).get('acc',[])
    if not L: print(f'{tag:>20} {"veri yok":>32}'); continue
    print(f'{tag:>20} {m(L):>10.3f}  [{min(L):.2f}-{max(L):.2f}] '
          f'{"(KONTROL)" if tag=="cubic_eta_default" else "":>9} {m(A):>10.1f}%  [{min(A):.0f}-{max(A):.0f}]')

print('\n=== HUKUM ===')
if len(base)<3:
    print('Kontrol kolu eksik -> hukum verilemez.')
else:
    best=None
    for tag in ['cubic_eta_m2_0','cubic_eta_0_p2','cubic_eta_m6_m4','cubic_eta_m8_m6']:
        L=results.get(tag,{}).get('loss',[])
        if len(L)<3: continue
        gain = m(base)-m(L)                       # pozitif = kayip dustu = iyilesme
        wins = sum(1 for a,b in zip(L,base) if a<b)
        print(f'{tag}: kazanc {gain:+.3f} nat, {wins}/3 seed varsayilandan iyi')
        if gain>=0.30 and wins==3:
            print(f'  -> CUBIC GELISTI: {tag} kriteri gecti (>=0.30 nat, 3/3). '
                  f'eta yanlis ayarliymis; yeni varsayilan adayi. Sonraki: graft rejiminde (§15h tekrari) sina.')
            best=best or tag
        elif gain>=0.15 and wins>=2:
            print(f'  -> KISMI: yon dogru ama esik altinda; daha genis tarama (-10..-8) gerekir.')
        else:
            print(f'  -> bu kol kriteri gecmedi.')
    if best is None:
        print('\nNULL ihtimali: hicbir kol >=0.30/3-3 gecmediyse eta ayari sorun degildi; '
              'cubic\'in siniri baska yerde. Durustce boyle raporlanir.')
    # exp ile kiyas (baglam)
    E=results.get('exp_reference',{}).get('loss',[])
    if E: print(f'\nBaglam: exp referans loss {m(E):.3f} | en iyi cubic '
                f'{min(m(results[t]["loss"]) for t in ["cubic_eta_default","cubic_eta_m6_m4","cubic_eta_m8_m6","cubic_eta_m2_0","cubic_eta_0_p2"] if results.get(t,{}).get("loss")):.3f}')